# Organize lines.csv into final_transcription folders

Reads `lines.csv` (image path + ground truth) and builds:

```
final_transcription/
  Book_5/
    14/  B5_P_014_l00.png  B5_P_014_l00.txt  ...
    15/  ...
  Book_6/
    14/  ...
```

- The `/content/drive/MyDrive` prefix in the CSV is ignored; images are read from the local `line_crops` folder next to this CSV.
- Each image keeps its original name; a sibling `.txt` with the same base name holds its ground truth.

In [ ]:
import os, re, csv, shutil
from pathlib import Path

# --- Config: point this at the folder that contains lines.csv and line_crops/ ---
BASE = Path('../transcriptions/colab/new/transcriptions-20260623T173303Z-3-001/transcriptions').resolve()
CSV_PATH = BASE / 'lines.csv'
OUTPUT_DIR = BASE / 'final_transcription'

print('CSV     :', CSV_PATH, '(exists:', CSV_PATH.exists(), ')')
print('Output  :', OUTPUT_DIR)

In [ ]:
# Map a CSV file_name to the real local image path, ignoring the Drive prefix.
# CSV value example: /content/drive/MyDrive/transcriptions/line_crops/5/B5_P_014_l00.png
def resolve_local_image(csv_path_value: str) -> Path:
    parts = Path(csv_path_value).parts
    # Keep everything from 'line_crops' onward and join to BASE.
    if 'line_crops' in parts:
        idx = parts.index('line_crops')
        return BASE.joinpath(*parts[idx:])
    # Fallback: just use the basename inside line_crops/<book>/
    return BASE / 'line_crops' / Path(csv_path_value).name

# Parse book number and page number from a filename like B5_P_014_l00.png
PATTERN = re.compile(r'B(\d+)_P_(\d+)_l\d+', re.IGNORECASE)

def parse_book_page(filename: str):
    m = PATTERN.match(Path(filename).name)
    if not m:
        return None, None
    book = int(m.group(1))
    page = int(m.group(2))  # '014' -> 14
    return book, page

In [ ]:
# Build the folder structure and copy image + write ground-truth txt.
written, missing, skipped = 0, [], []

with open(CSV_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        csv_path = (row.get('file_name') or '').strip()
        text = (row.get('text') or '').strip()
        if not csv_path:
            continue

        img_name = Path(csv_path).name           # keep original image name unchanged
        book, page = parse_book_page(img_name)
        if book is None:
            skipped.append(img_name)
            continue

        src_img = resolve_local_image(csv_path)
        if not src_img.exists():
            missing.append(str(src_img))
            continue

        dest_dir = OUTPUT_DIR / f'Book_{book}' / str(page)
        dest_dir.mkdir(parents=True, exist_ok=True)

        # Copy image (preserve name) and write matching .txt with same base name.
        shutil.copy2(src_img, dest_dir / img_name)
        (dest_dir / (Path(img_name).stem + '.txt')).write_text(text, encoding='utf-8')
        written += 1

print(f'Wrote {written} image+txt pairs into {OUTPUT_DIR}')
if missing:
    print(f'\n{len(missing)} images referenced in CSV but NOT found on disk:')
    for m in missing: print('   ', m)
if skipped:
    print(f'\n{len(skipped)} rows skipped (filename did not match B<book>_P_<page>_l<line>):')
    for s in skipped: print('   ', s)

In [ ]:
# Verify the result: count image/txt pairs per page folder.
for book_dir in sorted(OUTPUT_DIR.glob('Book_*')):
    print(book_dir.name)
    for page_dir in sorted(book_dir.iterdir(), key=lambda p: int(p.name) if p.name.isdigit() else p.name):
        pngs = list(page_dir.glob('*.png'))
        txts = list(page_dir.glob('*.txt'))
        flag = '' if len(pngs) == len(txts) else '  <-- MISMATCH'
        print(f'  page {page_dir.name}: {len(pngs)} png / {len(txts)} txt{flag}')